# Demo 24: Dynamic Programming and Policy Iteration / 動的計画法と方策反復

Use the same 5×5 maze to observe backward distance propagation, Bellman updates, and policy evaluation/improvement. / 同じ5×5迷路で、距離の逆向き伝播、ベルマン更新、方策評価と改善を観察する。

**How to use / 使い方：** Run the cells from top to bottom, then change the controls and compare the figures. / セルを上から順に実行し、操作項目を変えて図を比較する。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass
import ipywidgets as widgets
plt.rcParams.update({'figure.dpi': 105, 'font.size': 11})
np.set_printoptions(precision=3, suppress=True)

In [ ]:
# 共通の5×5迷路と設定 / Shared 5x5 maze and conventions.
# 0行目が上、0列目が左。SからGへの最短経路は8手 / Row 0 is top, column 0 is left; shortest S-to-G path is 8 steps.
N=5
START=(4,0); GOAL=(0,4)
WALLS=frozenset({(1,1),(2,1),(2,3),(3,3),(3,4)})
ACTIONS=[(-1,0),(1,0),(0,-1),(0,1)]
ACTION_NAMES=['Up','Down','Left','Right']
STATES=[(i,j) for i in range(N) for j in range(N) if (i,j) not in WALLS]
NONTERMINAL=[s for s in STATES if s!=GOAL]

def transition(s,a, step_cost=-0.04, goal_reward=1.0, walls=WALLS):
    if s==GOAL:return GOAL,0.0,True
    di,dj=ACTIONS[a];ns=(s[0]+di,s[1]+dj)
    if not(0<=ns[0]<N and 0<=ns[1]<N) or ns in walls:ns=s
    done=ns==GOAL
    return ns, goal_reward if done else step_cost, done

def render_maze(ax,values=None,policy=None,route=None,title='',walls=WALLS,fmt='.2f'):
    from matplotlib.patches import Rectangle
    ax.set_xlim(0,N);ax.set_ylim(N,0);ax.set_aspect('equal');ax.axis('off')
    valid=[v for s,v in (values or {}).items() if s not in walls and np.isfinite(v)]
    lo=min(valid) if valid else 0;hi=max(valid) if valid else 1
    for i in range(N):
        for j in range(N):
            s=(i,j)
            if s in walls:color='#626B7B'
            elif s==GOAL:color='#FAD9CA'
            elif s==START:color='#CDECFD'
            elif values is not None and s in values and hi>lo:
                color=plt.cm.YlGnBu(0.15+0.58*(values[s]-lo)/(hi-lo))
            else:color='#FFFFFF'
            ax.add_patch(Rectangle((j,i),1,1,facecolor=color,edgecolor='#8392A6',linewidth=1.1))
            if s in walls:continue
            if s==GOAL:label='G'
            elif values is not None and s in values:label=f'{values[s]:{fmt}}'
            elif s==START:label='S'
            else:label=''
            if label:ax.text(j+.5,i+.49,label,ha='center',va='center',color='#102E57',fontsize=9 if values is not None else 13,fontweight='bold')
            if policy is not None and s in policy and s!=GOAL and values is None:
                ax.text(j+.5,i+.72,'↑↓←→'[policy[s]],ha='center',va='center',color='#1763A8',fontsize=13)
    if route and len(route)>1:
        xx=[s[1]+.5 for s in route];yy=[s[0]+.5 for s in route]
        ax.plot(xx,yy,lw=3,color='#F39B2F',marker='o',markersize=4,alpha=.92)
    ax.set_title(title,fontweight='bold',fontsize=12)

## Part A — Backward shortest-path computation / その1：ゴールからの最短経路計算

Move cost is **1 per step**. Starting with a cost of **0 at the goal**, propagate minimum remaining costs to neighboring cells. / 移動コストは **1マスにつき1**。ゴールを **0** として、隣接マスに最小残りコストを順に伝える。

In [ ]:
from collections import deque

def min_distances(walls=WALLS):
    d={GOAL:0};q=deque([GOAL])
    while q:
        s=q.popleft()
        for di,dj in ACTIONS:
            n=(s[0]+di,s[1]+dj)
            if 0<=n[0]<N and 0<=n[1]<N and n not in walls and n not in d:
                d[n]=d[s]+1;q.append(n)
    return d

def shortest_route(walls=WALLS):
    d=min_distances(walls);s=START;route=[s]
    while s!=GOAL and len(route)<26:
        choices=[]
        for di,dj in ACTIONS:
            n=(s[0]+di,s[1]+dj)
            if n in d and d[n]<d[s]:choices.append(n)
        if not choices:break
        s=min(choices,key=lambda x:(d[x],x[0],x[1]));route.append(s)
    return route

d=min_distances();route=shortest_route()
print('Start distance / スタート距離 =',d[START],'| Path / 経路 =',route)
assert d[START]==8 and len(route)==9

In [ ]:
step=widgets.IntSlider(value=0,min=0,max=8,step=1,description='Distance / 距離',continuous_update=False)
show_path=widgets.Checkbox(value=True,description='Show path / 経路表示')
wall_selector=widgets.Dropdown(options=[(f'({i},{j})',(i,j)) for i in range(N) for j in range(N) if (i,j) not in (START,GOAL)],description='Cell / マス')
add_wall=widgets.Checkbox(value=False,description='Add wall / 壁を追加')
panel=widgets.Output()
def draw_dp(*_):
    walls=set(WALLS)
    if add_wall.value:walls.add(wall_selector.value)
    dd=min_distances(frozenset(walls));rr=shortest_route(frozenset(walls))
    visible={s:-v for s,v in dd.items() if v<=step.value}  # 色表示は負のコスト、数値表示は正の移動コスト / Color with negative cost, display positive movement cost.
    with panel:
        clear_output(wait=True)
        fig,axs=plt.subplots(1,2,figsize=(10,4))
        render_maze(axs[0],title='Maze / walls',walls=walls)
        if show_path.value and GOAL in dd and START in dd:render_maze(axs[0],route=rr,title='Maze and shortest route',walls=walls)
        # 距離伝播が到達したマスだけ数値を表示 / Display a value only after the distance wave reaches the cell.
        vals={s:float(v) for s,v in dd.items() if v<=step.value}
        render_maze(axs[1],values=vals,title=f'Computed distance ≤ {step.value}',walls=walls,fmt='.0f')
        plt.tight_layout();plt.show()
        print('Start distance / スタート距離:',dd.get(START,'unreachable'),'| computed cells / 計算済みマス:',len(vals))
for w in (step,show_path,wall_selector,add_wall):w.observe(draw_dp,names='value')
display(widgets.VBox([widgets.HBox([step,show_path]),widgets.HBox([wall_selector,add_wall]),panel]));draw_dp()

## Part B — Value and policy iteration / その2：価値反復と方策反復

- Normal step reward: **−0.04**; reward upon entering goal: **+1**; discount factor: **0.9**. / 通常移動の報酬は **−0.04**、ゴール到達報酬は **+1**、割引率は **0.9**。
- The episode ends at the goal, with **V(G) = 0**; attempting to enter a wall leaves the state unchanged and incurs the step reward. / ゴールで終了し **V(G) = 0**。壁への移動はその場に留まり、通常移動報酬を受ける。
- **Note:** Part A minimizes path cost; Part B maximizes discounted return. Both use Bellman-type recursion but are different objectives. / **注意：**その1は経路コストの最小化、その2は割引収益の最大化である。どちらもベルマン型の再帰計算だが、目的は異なる。

In [ ]:
def value_iteration(gamma=.9,alpha=None,iterations=180):
    v={s:0.0 for s in STATES};history=[v.copy()]
    for _ in range(iterations):
        vnew=v.copy()
        for s in NONTERMINAL:
            vnew[s]=max(transition(s,a)[1]+gamma*v[transition(s,a)[0]] for a in range(4))
        v=vnew;history.append(v.copy())
    policy={s:int(np.argmax([transition(s,a)[1]+gamma*v[transition(s,a)[0]] for a in range(4)])) for s in NONTERMINAL}
    return v,policy,history

def eval_policy(pi,gamma=.9,iterations=1000):
    v={s:0.0 for s in STATES}
    for _ in range(iterations):
        vn=v.copy()
        for s in NONTERMINAL:
            weights=pi[s]
            vn[s]=sum(weights[a]*(transition(s,a)[1]+gamma*v[transition(s,a)[0]]) for a in range(4))
        if max(abs(vn[s]-v[s]) for s in STATES)<1e-11:return vn
        v=vn
    return v

def iterate_policy(gamma=.9):
    pi={s:np.full(4,.25) for s in NONTERMINAL};samples=[]
    for k in range(40):
        v=eval_policy(pi,gamma);samples.append((k,{s:pi[s].copy() for s in pi},v.copy()))
        greedy={s:int(np.argmax([transition(s,a)[1]+gamma*v[transition(s,a)[0]] for a in range(4)])) for s in NONTERMINAL}
        newpi={s:np.eye(4)[greedy[s]] for s in NONTERMINAL}
        if all(np.allclose(newpi[s],pi[s]) for s in NONTERMINAL):break
        pi=newpi
    return samples

vstar,polstar,hist=value_iteration();samps=iterate_policy()
print('Value iteration / 価値反復:',round(vstar[START],3),'| Policy iteration / 方策反復:',round(samps[-1][2][START],3))
assert abs(vstar[START]-.270)<.002

In [ ]:
iteration_slider=widgets.IntSlider(value=0,min=0,max=30,description='VI step / 価値反復',continuous_update=False)
policy_slider=widgets.IntSlider(value=0,min=0,max=len(samps)-1,description='PI round / 方策反復',continuous_update=False)
mode=widgets.ToggleButtons(options=['Value iteration / 価値反復','Policy iteration / 方策反復'],description='Method / 手法')
output=widgets.Output()
def plot_iteration(*_):
    with output:
        clear_output(wait=True)
        fig,axs=plt.subplots(1,2,figsize=(10,4.2))
        if mode.value=='Value iteration / 価値反復':
            k=iteration_slider.value;v=hist[k];policy=polstar;heading=f'VI sweep {k}'
        else:
            k=policy_slider.value;v=samps[k][2]
            policy={s:int(np.argmax(samps[k][1][s])) for s in NONTERMINAL}
            heading=f'Policy iteration round {k}'
        render_maze(axs[0],values=v,title=heading)
        render_maze(axs[1],policy=policy,title='Greedy/current policy (arrows)')
        plt.tight_layout();plt.show()
        print('Start value / 初期状態価値:',round(v[START],5),'| Optimal / 最適値:',round(vstar[START],5))
for w in (mode,iteration_slider,policy_slider):w.observe(plot_iteration,names='value')
display(widgets.VBox([mode,iteration_slider,policy_slider,output]));plot_iteration()

## Try it yourself / 実験課題

1. Move **Distance / 距離** from 0 to 8 and explain how the start obtains its distance of 8. / 「距離」を0から8に動かし、スタートの距離8が求まる過程を説明する。
2. Block one additional cell and compare the shortest route and distance. / 壁を1マス追加し、最短経路と距離の変化を比較する。
3. Advance policy iteration and describe the changes in the start-state value. / 方策反復を進め、スタート状態の価値の変化を観察する。